# LAB 28 — Граф зон NYC Taxi: Dispatch Optimizer

**Модуль 3 · Python Advanced · Автор: Viktor Nikoriak**

---

## Що ми будуємо?

Ми будуємо **оптимізатор диспетчеризації таксі** на реальних даних NYC Yellow Taxi,
використовуючи теорію графів як основу.

**Сценарій:** Диспетчер отримує запит з зони 132 (Midtown Manhattan).
У нас є вільні машини в різних зонах міста.
Потрібно ранжувати їх за оптимальністю — хто доїде **найшвидше** і **найдешевше**.

**Центральне питання цієї лабораторії:**
> Як реальні дані про поїздки перетворюються на **граф зон**?  
> Коли використовувати BFS (мін. переходів), а коли Dijkstra (мін. вартість)?  
> Як той самий запит виглядає у **Neo4j Cypher**?

---

## Карта лабораторії

| Частина | Тема | Алгоритм |
|---------|------|----------|
| 1 | Завантаження даних (DuckDB Lazy Architecture) | — |
| 2 | Побудова графу зон з реальних поїздок | Adjacency List |
| 3 | BFS Dispatch — мінімум переходів між зонами | BFS |
| 4 | Dijkstra Dispatch — найдешевший маршрут | Dijkstra |
| 5 | Neo4j — реальне підключення і виконання Cypher | Cypher |
| 6 | Візуалізація зв'язності зон | Matplotlib |
| 7 | Фінальний Dispatch Optimizer | Інтеграція |

---

> **Принцип:** Граф — це не абстракція. Це спосіб побачити у таблиці поїздок
> **приховану мережу зв'язків** між зонами міста.

## Частина 1: Завантаження Даних — DuckDB Lazy Architecture

### Архітектура шарів

```
NYC Parquet (3M+ рядків, хмара)
        ↓  HTTPS + httpfs extension
DuckDB VIEW «trips»   ← визначає ЯК читати, але не читає нічого
        ↓  SQL SAMPLE {n} ROWS
pandas DataFrame      ← лише n рядків у RAM (~5 000)
        ↓  aggregation
Python dict (граф)   ← {зона: [(сусід, вага), ...]}
```

**Колонки, які нас цікавлять:**

| Колонка | Тип | Призначення |
|---------|-----|-------------|
| `pickup_dt` | datetime | Час виклику |
| `trip_distance` | float | Відстань поїздки (милі) |
| `total_amount` | float | Загальна сума (USD) — вага ребра |
| `pu_zone` | int | Зона посадки 1–263 — вузол графу |
| `do_zone` | int | Зона висадки 1–263 — вузол графу |
| `passengers` | int | Кількість пасажирів |

In [ ]:
# ── Залежності ─────────────────────────────────────────────────────────────
# pip install duckdb pandas matplotlib neo4j

import time
import heapq
import random
from collections import deque, defaultdict

import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

In [ ]:
# ── DuckDB: підключення в пам'яті ─────────────────────────────────────────
con = duckdb.connect(database=":memory:")

try:
    con.execute("INSTALL httpfs; LOAD httpfs;")
    print("httpfs завантажено")
except Exception as e:
    print(f"  httpfs вже присутній: {e}")

In [ ]:
# ── Lazy VIEW ─────────────────────────────────────────────────────────────
DATA_URL = (
    "https://d37ci6vzurychx.cloudfront.net/trip-data/"
    "yellow_tripdata_2023-01.parquet"
)

con.execute(f"""
    CREATE OR REPLACE VIEW trips AS
    SELECT
        tpep_pickup_datetime                     AS pickup_dt,
        CAST(trip_distance   AS DOUBLE)          AS trip_distance,
        CAST(fare_amount     AS DOUBLE)          AS fare_amount,
        CAST(tip_amount      AS DOUBLE)          AS tip_amount,
        CAST(total_amount    AS DOUBLE)          AS total_amount,
        CAST(PULocationID    AS INTEGER)         AS pu_zone,
        CAST(DOLocationID    AS INTEGER)         AS do_zone,
        CAST(passenger_count AS INTEGER)         AS passengers
    FROM read_parquet('{DATA_URL}')
    WHERE
        trip_distance > 0.1 AND trip_distance < 100
        AND fare_amount   > 0 AND fare_amount   < 500
        AND total_amount  > 0
        AND passenger_count >= 1
""")

print("VIEW 'trips' зареєстровано (lazy — жодних даних ще в RAM)")

In [ ]:
# ── Завантаження ЗРАЗКУ поїздок ────────────────────────────────────────────
SAMPLE_SIZE = 5_000

t0 = time.time()
df_trips = con.execute(f"""
    SELECT *
    FROM trips
    USING SAMPLE {SAMPLE_SIZE} ROWS
""").df()
elapsed = time.time() - t0

print(f"Завантажено: {len(df_trips):,} поїздок за {elapsed:.2f}с")
print(f"RAM для DataFrame: {df_trips.memory_usage(deep=True).sum() / 1024:.1f} KB")
print(f"\nПерші 3 рядки:")
df_trips.head(3)

## Частина 2: Побудова Графу Зон

### Концепція відображення

```
DataFrame рядок:  pu_zone=132, do_zone=138, total_amount=45.30
         ↓
Граф:     вузол 132 → вузол 138, вага = середній total_amount
```

**Ми будуємо два графи:**
1. **Незважений** (для BFS) — зона A → зона B якщо є хоч одна поїздка
2. **Зважений** (для Dijkstra) — вага = середня вартість поїздки між зонами

**Ребро існує якщо:** між двома зонами відбулося щонайменше **MIN_TRIPS** поїздок.

In [ ]:
# ── Агрегація: зона-до-зони ────────────────────────────────────────────────
MIN_TRIPS = 3

zone_pairs = (
    df_trips
    .groupby(["pu_zone", "do_zone"])
    .agg(
        trip_count=("total_amount", "count"),
        avg_cost=("total_amount", "mean"),
        avg_distance=("trip_distance", "mean"),
    )
    .reset_index()
    .query("trip_count >= @MIN_TRIPS")
    .query("pu_zone != do_zone")
)

print(f"Пар зон з >= {MIN_TRIPS} поїздками: {len(zone_pairs)}")
print(f"Топ-5 найпопулярніших маршрутів:")
zone_pairs.nlargest(5, "trip_count")[["pu_zone","do_zone","trip_count","avg_cost"]]

In [ ]:
# ── Побудова графу (adjacency list) ───────────────────────────────────────
graph_unweighted = defaultdict(list)
graph_weighted   = defaultdict(list)

for _, row in zone_pairs.iterrows():
    src  = int(row["pu_zone"])
    dst  = int(row["do_zone"])
    cost = float(row["avg_cost"])
    graph_unweighted[src].append(dst)
    graph_weighted[src].append((dst, cost))

graph_unweighted = dict(graph_unweighted)
graph_weighted   = dict(graph_weighted)

total_zones = len(graph_unweighted)
total_edges = sum(len(n) for n in graph_unweighted.values())

print(f"=== Граф зон NYC Taxi ===")
print(f"Вузлів |V| = {total_zones} зон")
print(f"Ребер  |E| = {total_edges} маршрутів")
print(f"Середній out-degree = {total_edges/total_zones:.1f} зон/вузол")
print(f"Density = {total_edges / (total_zones * (total_zones-1)) * 100:.2f}%")

## Частина 3: BFS Dispatch — найближча зона за кількістю переходів

### Задача диспетчера

Є запит з **зони 132** (Midtown Manhattan).
BFS відповідає на питання: **«Яка зона знаходиться за мінімальну кількість переходів?»**

### Коли BFS підходить, а коли ні?

- ✅ **BFS підходить**: усі зони «однаково далеко» → підрахунок переходів має сенс
- ❌ **BFS не підходить**: зони мають різну відстань/вартість → потрібен Dijkstra

In [ ]:
def bfs_dispatch(graph: dict, request_zone: int, max_hops: int = 5) -> list[dict]:
    result = []
    visited = {request_zone}
    queue = deque([(request_zone, 0, [request_zone])])

    while queue:
        zone, hops, path = queue.popleft()
        if zone != request_zone:
            result.append({"zone": zone, "hops": hops, "path": path})
        if hops >= max_hops:
            continue
        for neighbor in graph.get(zone, []):
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, hops + 1, path + [neighbor]))

    return sorted(result, key=lambda x: x["hops"])


REQUEST_ZONE = 132
if REQUEST_ZONE not in graph_unweighted:
    REQUEST_ZONE = max(graph_unweighted, key=lambda z: len(graph_unweighted.get(z, [])))
    print(f"Зона 132 не в зразку. Використовуємо зону {REQUEST_ZONE}")

bfs_zones = bfs_dispatch(graph_unweighted, REQUEST_ZONE, max_hops=3)

print(f"=== BFS Dispatch: запит з Zone {REQUEST_ZONE} ===")
print(f"{'Зона':>6}  {'Хопи':>6}  {'Маршрут':<40}")
print("-" * 60)
for entry in bfs_zones[:15]:
    print(f"Zone {entry['zone']:>3}  {entry['hops']:>6}  {' → '.join(map(str, entry['path']))}")
print(f"\nВсього досяжних зон (max 3 хопи): {len(bfs_zones)}")

## Частина 4: Dijkstra Dispatch — найдешевший маршрут

### Чому Dijkstra краще для реального диспетчера?

BFS вважає, що всі переходи між зонами **однаково дорогі**.  
Але реально:
- Зона 132 → 138 (Midtown → JFK): середня поїздка **$45**
- Зона 132 → 236 (Midtown → Upper West): середня поїздка **$12**

Dijkstra враховує ці різниці і знаходить маршрут з **мінімальною загальною вартістю**.

> **Аналогія з GPS:** BFS = «мінімум поворотів», Dijkstra = «мінімум часу/палива».

In [ ]:
def dijkstra_dispatch(graph: dict, start: int) -> tuple[dict, dict]:
    distances    = {node: float('inf') for node in graph}
    distances[start] = 0.0
    predecessors = {node: None for node in graph}
    heap    = [(0.0, start)]
    visited = set()

    while heap:
        current_cost, current_zone = heapq.heappop(heap)
        if current_zone in visited:
            continue
        visited.add(current_zone)
        for neighbor_zone, edge_cost in graph.get(current_zone, []):
            new_cost = current_cost + edge_cost
            if new_cost < distances.get(neighbor_zone, float('inf')):
                distances[neighbor_zone]    = new_cost
                predecessors[neighbor_zone] = current_zone
                heapq.heappush(heap, (new_cost, neighbor_zone))

    return distances, predecessors


def get_optimal_path(predecessors: dict, start: int, goal: int) -> list:
    path, node = [], goal
    while node is not None:
        path.append(node)
        node = predecessors.get(node)
        if node == start:
            path.append(start)
            break
    path.reverse()
    return path if path and path[0] == start else [goal]


dist, pred = dijkstra_dispatch(graph_weighted, REQUEST_ZONE)

reachable = sorted(
    [(z, c) for z, c in dist.items() if c < float('inf') and z != REQUEST_ZONE],
    key=lambda x: x[1]
)

print(f"=== Dijkstra Dispatch: найдешевші маршрути з Zone {REQUEST_ZONE} ===")
print(f"{'Зона':>6}  {'Мін. вартість':>14}  {'Маршрут':<50}")
print("-" * 75)
for zone, cost in reachable[:15]:
    path = get_optimal_path(pred, REQUEST_ZONE, zone)
    print(f"Zone {zone:>3}  ${cost:>12.1f}  {' → '.join(map(str, path))}")

---
## Частина 5: Neo4j — Реальне Підключення і Cypher

До цього моменту граф існував лише **в пам'яті Python** (adjacency list у dict).  
Тепер ми завантажимо **ті самі дані** (`zone_pairs`) у Neo4j і виконаємо
BFS і Dijkstra через **Cypher** — мову запитів до графової бази.

### Що змінюється?

| | Python (dict) | Neo4j (граф БД) |
|--|--|--|
| Де зберігається | RAM (зникає після зупинки) | Диск (постійно) |
| Мова запитів | Python BFS/Dijkstra | Cypher `MATCH` |
| Масштаб | ~тисячі вузлів | мільярди вузлів |
| Multi-hop traversal | loops + recursion | `*1..3` pattern |

### Граф-модель

```
(:Zone {zone_id})-[:CONNECTED_TO {freq, avg_cost, weight}]->(:Zone)
```

### Docker ports (docker-compose.yml)
```
7474:7474  →  http://localhost:7474   (Neo4j Browser)
7687:7687  →  bolt://localhost:7687   (Python driver)
```

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI  = "bolt://localhost:7687"
NEO4J_AUTH = ("neo4j", "python2024")

neo4j_driver = GraphDatabase.driver(NEO4J_URI, auth=NEO4J_AUTH)
neo4j_driver.verify_connectivity()
print("✓ Neo4j connected")

In [ ]:
# ── Очистити і створити схему ──────────────────────────────────────────────
with neo4j_driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    session.run("CREATE CONSTRAINT zone_id IF NOT EXISTS "
                "FOR (z:Zone) REQUIRE z.zone_id IS UNIQUE")
print("✓ Schema ready")

In [ ]:
# ── Вставляємо Zone вузли і CONNECTED_TO ребра з zone_pairs ───────────────
# Дані вже є у пам'яті (Part 2) — просто завантажуємо їх у Neo4j

all_zone_ids = sorted(
    set(zone_pairs['pu_zone'].tolist() + zone_pairs['do_zone'].tolist())
)
zone_records = [{"zone_id": int(z)} for z in all_zone_ids]

edge_records = [
    {
        "pu_zone":  int(r.pu_zone),
        "do_zone":  int(r.do_zone),
        "freq":     int(r.trip_count),
        "avg_cost": round(float(r.avg_cost), 2),
        "avg_dist": round(float(r.avg_distance), 2),
        "weight":   round(1.0 / float(r.trip_count), 4),
    }
    for r in zone_pairs.itertuples()
]

CREATE_ZONES = "UNWIND $zones AS z MERGE (:Zone {zone_id: z.zone_id})"
CREATE_EDGES = """
UNWIND $edges AS e
MATCH (src:Zone {zone_id: e.pu_zone})
MATCH (dst:Zone {zone_id: e.do_zone})
MERGE (src)-[r:CONNECTED_TO]->(dst)
  SET r.freq     = e.freq,
      r.avg_cost = e.avg_cost,
      r.avg_dist = e.avg_dist,
      r.weight   = e.weight
"""

with neo4j_driver.session() as session:
    session.run(CREATE_ZONES, zones=zone_records)
    for i in range(0, len(edge_records), 300):
        session.run(CREATE_EDGES, edges=edge_records[i:i+300])

print(f"✓ {len(zone_records)} Zone nodes")
print(f"✓ {len(edge_records)} CONNECTED_TO edges")

### 5.1 — BFS у Cypher

Патерн `[:CONNECTED_TO*1..3]` — це Cypher-еквівалент нашого Python BFS.  
Neo4j обходить граф через **index-free adjacency**: кожен вузол зберігає
прямий pointer на сусідів — O(1) на кожен hop, незалежно від розміру графу.

```cypher
MATCH (start:Zone {zone_id: 132})-[:CONNECTED_TO*1..3]->(end:Zone)
```

In [ ]:
Q_BFS = """
MATCH path = (start:Zone {zone_id: $zone})-[:CONNECTED_TO*1..3]->(end:Zone)
WITH end.zone_id AS zone, length(path) AS hops
RETURN zone, min(hops) AS hops
ORDER BY hops ASC, zone ASC
LIMIT 15
"""

with neo4j_driver.session() as session:
    bfs_cypher = session.run(Q_BFS, zone=REQUEST_ZONE).data()

print(f"BFS (Cypher) від Zone {REQUEST_ZONE}:")
print(f"{'Zone':>6}  {'Hops':>6}")
print("-" * 18)
for r in bfs_cypher:
    print(f"Zone {r['zone']:>3}  {r['hops']:>6}")

# Порівняння з Python BFS
python_bfs_set = {e['zone'] for e in bfs_zones}
cypher_bfs_set = {r['zone'] for r in bfs_cypher}
print(f"\nPython BFS: {len(python_bfs_set)} зон | Cypher BFS: {len(cypher_bfs_set)} зон")
print(f"Спільних: {len(python_bfs_set & cypher_bfs_set)}")

### 5.2 — Зважений шлях у Cypher (Dijkstra-аналог)

`REDUCE` підраховує накопичену вартість по ребрах шляху —  
той самий принцип що і наш Python Dijkstra (relaxation по вагах).

```cypher
REDUCE(cost = 0.0, r IN rels | cost + r.avg_cost) AS total_cost
```

In [ ]:
Q_WEIGHTED = """
MATCH path = (start:Zone {zone_id: $zone})
      -[rels:CONNECTED_TO*1..3]->
      (end:Zone)

WHERE ALL(n IN nodes(path) WHERE single(x IN nodes(path) WHERE x = n))

WITH end, path, rels,
     REDUCE(cost = 0.0, r IN rels | cost + r.avg_cost) AS total_cost

RETURN end.zone_id AS zone,
       ROUND(total_cost, 1) AS min_cost,
       [n IN nodes(path) | n.zone_id] AS route

ORDER BY total_cost ASC
LIMIT 15
"""

with neo4j_driver.session() as session:
    dijkstra_cypher = session.run(Q_WEIGHTED, zone=REQUEST_ZONE).data()

print(f"Weighted path (Cypher) від Zone {REQUEST_ZONE}:")
print(f"{'Zone':>6}  {'Cost ($)':>10}  Route")
print("-" * 55)
for r in dijkstra_cypher:
    route_str = " → ".join(map(str, r['route']))
    print(f"Zone {r['zone']:>3}  ${r['min_cost']:>9.1f}  {route_str}")

In [ ]:
# ── Аналітичні Cypher запити ───────────────────────────────────────────────

# Q1: Топ-10 найбільш зв'язаних зон (out-degree)
Q_TOP = """
MATCH (z:Zone)-[r:CONNECTED_TO]->()
RETURN z.zone_id AS zone, COUNT(r) AS connections, SUM(r.freq) AS total_trips
ORDER BY total_trips DESC LIMIT 10
"""

# Q2: Найзавантаженіші коридори зона→зона
Q_CORRIDORS = """
MATCH (src:Zone)-[r:CONNECTED_TO]->(dst:Zone)
RETURN src.zone_id AS from_zone, dst.zone_id AS to_zone,
       r.freq AS trips, r.avg_cost AS avg_cost
ORDER BY r.freq DESC LIMIT 8
"""

with neo4j_driver.session() as session:
    print("Top-10 зон за виходами:")
    for r in session.run(Q_TOP).data():
        print(f"  zone={r['zone']:3d} | connections={r['connections']:3d} | trips={r['total_trips']}")

    print("\nНайзавантаженіші коридори:")
    for r in session.run(Q_CORRIDORS).data():
        print(f"  {r['from_zone']:3d} → {r['to_zone']:3d} | trips={r['trips']} | avg=${r['avg_cost']:.1f}")

### Python vs Cypher — підсумок

| Операція | Python (dict + BFS/Dijkstra) | Cypher |
|----------|------------------------------|--------|
| BFS до глибини 3 | `deque` loop, ~20 рядків | `[:CONNECTED_TO*1..3]` |
| Зважений шлях | `heapq` + relaxation | `REDUCE(cost + r.avg_cost)` |
| Top-N зон | `sorted(dict.items())` | `ORDER BY ... LIMIT` |
| Зберігання | RAM, зникає | Диск, постійно |
| Масштаб | тисячі вузлів | мільярди |

> **Коли Python:** прототипування, невелика вибірка, дослідження алгоритму.  
> **Коли Neo4j:** production, великий граф, складні multi-hop запити.

---
## Частина 6: Візуалізація — зв'язність зон

### Що ми хочемо побачити?

1. **Розподіл out-degree:** скільки унікальних зон призначення має кожна зона
2. **Топ-20 зон за кількістю виходів:** найпопулярніші pickup-зони
3. **Розподіл ваг ребер:** медіана, відхилення вартостей між зонами

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("NYC Taxi Zone Graph — аналіз структури", fontsize=14, fontweight="bold")

# Графік 1: розподіл out-degree
ax1 = axes[0]
out_degrees = [len(neighbors) for neighbors in graph_unweighted.values()]
ax1.hist(out_degrees, bins=20, color="#1565C0", edgecolor="white", alpha=0.85)
mean_deg = sum(out_degrees) / len(out_degrees)
ax1.axvline(x=mean_deg, color="#FF6F00", linewidth=2, linestyle="--",
            label=f"Середнє: {mean_deg:.1f}")
ax1.set_title("Розподіл out-degree")
ax1.set_xlabel("Кількість зон-призначень")
ax1.set_ylabel("Кількість зон")
ax1.legend()

# Графік 2: топ-20 зон за out-degree
ax2 = axes[1]
zone_degree = sorted(
    [(z, len(n)) for z, n in graph_unweighted.items()],
    key=lambda x: x[1], reverse=True
)[:20]
zones_top, degrees_top = zip(*zone_degree)
ax2.barh([str(z) for z in zones_top], degrees_top, color="#2E7D32", edgecolor="white")
ax2.set_title("Топ-20 зон за out-degree")
ax2.set_xlabel("Out-degree")
ax2.set_ylabel("Zone ID")
ax2.invert_yaxis()

# Графік 3: розподіл ваг ребер
ax3 = axes[2]
all_costs = zone_pairs["avg_cost"].values
ax3.hist(all_costs, bins=30, color="#6A1B9A", edgecolor="white", alpha=0.85)
median_cost = float(zone_pairs["avg_cost"].median())
ax3.axvline(x=median_cost, color="#FF6F00", linewidth=2, linestyle="--",
            label=f"Медіана: ${median_cost:.1f}")
ax3.set_title("Розподіл середніх вартостей (ваги ребер)")
ax3.set_xlabel("Середня вартість поїздки ($)")
ax3.set_ylabel("Кількість пар зон")
ax3.legend()

plt.tight_layout()
plt.show()

print(f"\nКлючові метрики графу:")
print(f"  Найбільший хаб: Zone {zone_degree[0][0]} ({zone_degree[0][1]} маршрутів)")
print(f"  Медіана ваги ребра: ${median_cost:.1f}")
print(f"  Мін/Макс вага: ${all_costs.min():.1f} / ${all_costs.max():.1f}")

In [ ]:
# ── Графік 4: BFS vs Dijkstra — порівняння ранжування ─────────────────────
bfs_top20      = {e["zone"]: e["hops"] for e in bfs_zones[:20]}
dijkstra_top20 = {zone: cost for zone, cost in reachable[:20]}
common_zones   = set(bfs_top20.keys()) & set(dijkstra_top20.keys())

if common_zones:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"BFS vs Dijkstra — ранжування від Zone {REQUEST_ZONE}",
                 fontsize=13, fontweight="bold")

    bfs_data = sorted(bfs_top20.items(), key=lambda x: x[1])[:15]
    bfs_labels, bfs_values = zip(*bfs_data) if bfs_data else ([], [])
    axes[0].barh([f"Zone {z}" for z in bfs_labels], bfs_values,
                 color="#1565C0", edgecolor="white")
    axes[0].set_title("BFS: мінімум переходів")
    axes[0].set_xlabel("Кількість хопів")
    axes[0].invert_yaxis()

    dij_data = sorted(dijkstra_top20.items(), key=lambda x: x[1])[:15]
    dij_labels, dij_values = zip(*dij_data) if dij_data else ([], [])
    axes[1].barh([f"Zone {z}" for z in dij_labels], dij_values,
                 color="#2E7D32", edgecolor="white")
    axes[1].set_title("Dijkstra: мінімальна вартість ($)")
    axes[1].set_xlabel("Середня вартість маршруту ($)")
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()
else:
    print("Збільшіть SAMPLE_SIZE до 50_000 для повнішого графу.")

---
## Частина 7: Фінальний Dispatch Optimizer

### Інтеграція всього

Диспетчер отримує:
- `request_zone` — зону виклику
- `available_cars` — словник `{зона: кількість_вільних_машин}`

Алгоритм:
1. **Dijkstra** від `request_zone` → вартість до кожної зони
2. **Збіг** — перетинаємо досяжні зони з `available_cars`
3. **Ранжування** — сортуємо за мін. вартістю маршруту
4. **Dispatch** — перша машина у списку отримує виклик

In [ ]:
def dispatch_optimizer(
    graph_w: dict,
    graph_u: dict,
    request_zone: int,
    available_cars: dict,
    strategy: str = "cost",
) -> list[dict]:
    candidates = []

    if strategy == "cost":
        d, p = dijkstra_dispatch(graph_w, request_zone)
        for zone, cars in available_cars.items():
            if zone in d and d[zone] < float('inf'):
                candidates.append({
                    "zone": zone, "cars": cars,
                    "score": round(d[zone], 2),
                    "path": get_optimal_path(p, request_zone, zone),
                    "strategy": "Dijkstra (avg $)",
                })
        candidates.sort(key=lambda x: x["score"])

    elif strategy == "hops":
        bfs_lookup = {e["zone"]: e for e in bfs_dispatch(graph_u, request_zone, max_hops=10)}
        for zone, cars in available_cars.items():
            if zone in bfs_lookup:
                e = bfs_lookup[zone]
                candidates.append({
                    "zone": zone, "cars": cars,
                    "score": e["hops"], "path": e["path"],
                    "strategy": "BFS (хопи)",
                })
        candidates.sort(key=lambda x: x["score"])

    return candidates


random.seed(42)
available_cars = {
    zone: random.randint(1, 4)
    for zone in random.sample(list(graph_weighted.keys()),
                              min(20, len(graph_weighted)))
}

print(f"=== Виклик з Zone {REQUEST_ZONE} ===")
print(f"Вільні машини: {len(available_cars)} зон, {sum(available_cars.values())} машин\n")

for strategy in ["cost", "hops"]:
    result = dispatch_optimizer(graph_weighted, graph_unweighted,
                                REQUEST_ZONE, available_cars, strategy)
    label = "Вартість ($)" if strategy == "cost" else "Хопів"
    print(f"--- Стратегія: {strategy.upper()} ---")
    print(f"{'Зона':>6}  {'Машин':>6}  {label:>14}  {'Маршрут':<40}")
    print("-" * 75)
    for car in result[:8]:
        path_str = " → ".join(map(str, car['path']))
        print(f"Zone {car['zone']:>3}  {car['cars']:>6}  {car['score']:>14.1f}  {path_str}")
    if result:
        best = result[0]
        print(f"\n✅ DISPATCH: Zone {best['zone']} ({best['cars']} машини, {label}: {best['score']:.1f})")
    print()

In [ ]:
# Закриваємо з'єднання з Neo4j
neo4j_driver.close()
con.close()
print("Connections closed.")

---
## Підсумок: три шари одного сценарію

```
┌─────────────────────────────────────────────────────────────┐
│  Реальні дані (NYC Parquet, 3M+ рядків)                     │
│           ↓ DuckDB lazy VIEW → SAMPLE 5000                  │
│  pandas DataFrame → агрегація по парах (zone_a, zone_b)     │
│           ↓                                                 │
│  Python Adjacency List                                      │
│  ├─ graph_unweighted → BFS (мін. хопів)                     │
│  └─ graph_weighted   → Dijkstra (мін. вартість)             │
│           ↓                                                 │
│  Neo4j (:Zone)-[:CONNECTED_TO]->(:Zone)                     │
│  ├─ [:CONNECTED_TO*1..3]   → BFS у Cypher                   │
│  └─ REDUCE(cost + r.avg_cost) → Dijkstra у Cypher           │
│           ↓                                                 │
│  dispatch_optimizer() → ранжований список машин             │
└─────────────────────────────────────────────────────────────┘
```

### Ключові інсайти

| Питання | Відповідь |
|---------|----------|
| Чому граф, а не sorted list? | Traversal на кілька хопів, непрямі маршрути |
| BFS vs Dijkstra? | BFS = «скільки зон між нами», Dijkstra = «скільки це коштує» |
| Чому Neo4j > SQL для цього? | SQL JOINs деградують O(n³), Neo4j traversal = O(depth) |
| Python vs Neo4j? | Python = прототип в пам'яті, Neo4j = продакшн на мільярдах |

### Що далі?

- **PageRank** для вузлів: які зони є найбільшими «хабами» по NYC?
- **Louvain community detection**: автоматично знайти «райони» без знання мапи
- **A\* Search**: додати геокоординати зон як евристику для ще швидшого пошуку

---


## Як розвинути цей прототип?

#### 1. Real-time Driver Location
```python
# Замість статичного LOCATED_IN — оновлення кожні 10 сек
# WebSocket → Redis Pub/Sub → Neo4j SET drv.zone_id = $zone
driver.execute_query(
    "MATCH (d:Driver {driver_id: $id})-[r:LOCATED_IN]->() "
    "DELETE r "
    "WITH d MATCH (z:Zone {zone_id: $new_zone}) "
    "CREATE (d)-[:LOCATED_IN]->(z)",
    id=driver_id, new_zone=new_zone
)
```

#### 2. Streaming Trips (Kafka / Redis Streams)
```
GPS Event → Redis Stream → Consumer → Neo4j MERGE
                                   → Edge weight recalculation
```

#### 3. FastAPI Microservice
```python
# GET /dispatch?zone=161
@app.get("/dispatch")
async def dispatch(zone: int):
    with driver.session() as s:
        rows = s.run(Q_APPROACH3, zone=zone).data()
    return {"best_driver": rows[0] if rows else None}
```

#### 4. Recommendation System
```
[:CONNECTED_TO] ребра з вагами = graph embeddings
→ Де водій має найвищий заробіток? (Node2Vec → ML)
→ Яка зона next-best після dropoff? (PageRank)
```

---
## 🔥 BONUS — From Data to System

### Як цей ноутбук стає production системою?

```
PROTOTYPE (this notebook)
        │
        ▼
┌─────────────────────────────────────────────────┐
│             PRODUCTION ARCHITECTURE             │
│                                                 │
│  GPS Events ──► Kafka ──► Stream Processor      │
│                                │                │
│                                ▼                │
│                         Neo4j (Graph DB)        │
│                         Zone graph + Drivers    │
│                                │                │
│                                ▼                │
│  Client App ──► FastAPI ──► Dispatch Engine     │
│                    │        (Dijkstra query)    │
│                    │                            │
│                    ▼                            │
│               Redis Cache ◄── Response          │
└─────────────────────────────────────────────────┘
```

### Три шари трансформації

| Шар | Ноутбук | Production |
|-----|---------|------------|
| **Дані** | Pandas DataFrame | Kafka topic + PostgreSQL WAL |
| **Граф** | Neo4j (local Docker) | Neo4j Aura Enterprise / cluster |
| **API** | Cypher в ноутбуці | FastAPI + connection pool |

### Чому Neo4j, а не PostgreSQL для диспетчеризації?

```sql
-- PostgreSQL: shortest path між 2 зонами
WITH RECURSIVE path AS (
  SELECT zone_from, zone_to, 1 AS depth, ARRAY[zone_from] AS visited
  FROM zone_connections WHERE zone_from = 161
  UNION ALL
  SELECT c.zone_from, c.zone_to, p.depth + 1, p.visited || c.zone_from
  FROM zone_connections c JOIN path p ON c.zone_from = p.zone_to
  WHERE NOT c.zone_from = ANY(p.visited) AND p.depth < 3
)
SELECT * FROM path WHERE zone_to IN (SELECT zone FROM drivers WHERE status='free');
-- 15 рядків коду, SQL JOIN hell, важко підтримувати
```

```cypher
// Neo4j: той самий запит
MATCH (z:Zone {zone_id:161})-[:CONNECTED_TO*1..3]-(n:Zone)
      <-[:LOCATED_IN]-(d:Driver {status:'free'})
RETURN d, length(path) ORDER BY length(path)
// 3 рядки, читається як речення
```

> **Висновок:** графова БД — не просто технологія. Це спосіб думати про задачу.
> Коли твоя предметна область — **відносини** між об'єктами, граф — природній вибір.

# Самостійні задачі (Neo4j)

## 🟢 Задача 1: Найпопулярніші напрямки з однієї зони

🎯 Умова:

Вибери будь-яку зону (наприклад `zone_id = 48`)
і знайди ТОП-5 зон, куди з неї найчастіше їдуть.

📌 Що потрібно:

- використати `CONNECTED_TO`
- відсортувати по `freq`
- показати:
  - zone_id
  - кількість поїздок

📥 Очікуваний результат:

48 → 161 (120 trips)
48 → 90  (98 trips)
...

🧠 Підказка:

MATCH (a:Zone {zone_id: ...})-[r:CONNECTED_TO]->(b:Zone)

+ ORDER BY r.freq DESC
+ LIMIT 5

___
### 💡 Що це дає

* розуміння **локального графа**
* як працює **ranking на ребрах**







## 🟡 Задача 2: Зони без виходів (dead ends)

🎯 Умова:

Знайди всі зони, з яких **немає вихідних маршрутів**.

📌 Що потрібно:

- знайти вузли `Zone`
- у яких НЕМАЄ `CONNECTED_TO` назовні

📥 Очікуваний результат:

zone_id: 205
zone_id: 177
...

🧠 Підказка:

MATCH (z:Zone)
WHERE NOT (z)-[:CONNECTED_TO]->()
RETURN z

___
### 💡 Що це дає

* розуміння **структури графа**
* знаходження “мертвих точок”
* intro до **graph validation**





## 🔴 Задача 3: Найдешевший маршрут (спрощений Dijkstra)

🎯 Умова:

Знайди маршрут від зони `48` до `161`,
який має **найменшу сумарну вартість (avg_cost)**.

📌 Обмеження:

- глибина до 3–4 кроків
- використовувати `REDUCE`

📥 Очікуваний результат:

48 → 90 → 161
Total cost: 24.5

🧠 Підказка:

MATCH p = (:Zone {zone_id: 48})-[r:CONNECTED_TO*1..4]->(:Zone {zone_id: 161})

WITH p,
     REDUCE(cost = 0.0, rel IN r | cost + rel.avg_cost) AS total_cost

RETURN p, total_cost
ORDER BY total_cost ASC
LIMIT 1

___

### 💡 Що це дає

* розуміння **вагового графа**
* як працює логіка Dijkstra
* перехід від “зв’язків” до **оптимізації**
